In [90]:
from dotenv import load_dotenv
import os
from langchain_neo4j import Neo4jGraph

from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate
from pydantic import BaseModel, Field
from typing import Tuple, List
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import WikipediaLoader
from langchain.text_splitter import TokenTextSplitter
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer

from langchain_neo4j import Neo4jVector
from langchain_openai import OpenAIEmbeddings
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars

In [91]:
load_dotenv(override=True)

True

In [92]:
AURA_INSTANCENAME = os.environ["AURA_INSTANCENAME"]
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_ENDPOINT = os.getenv("OPENAI_ENDPOINT")

In [87]:
chat = ChatOpenAI(api_key=OPENAI_API_KEY, temperature=0, model="gpt-4o-mini")

In [93]:
kg = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
) 

In [94]:
# # # read the wikipedia page for the Roman Empire
# raw_documents = WikipediaLoader(query="The Roman empire").load()

from langchain_core.documents import Document
raw_documents = [
    Document(
        page_content="""
        The Maratha Empire, also referred to as the Maratha Confederacy, was an early modern polity in the Indian subcontinent. For most of its existence, it comprised the realms of the Peshwa and four major independent Maratha states under the nominal leadership of the former and nominal loyalty to the Chhatrapatis who were successors of Chhatrapati Shivaji Maharaj.

        The Marathas were a Marathi-speaking warrior group from the western Deccan Plateau (present-day Maharashtra) that rose to prominence under leadership of Shivaji (17th century), who revolted against the Bijapur Sultanate and the Mughal Empire for establishing "Hindavi Swarajya" (lit. 'self-rule of Hindus').

        The religious attitude of Emperor Aurangzeb estranged non-Muslims, and the Maratha insurgency came at a great cost for his men and treasury. The Maratha government also included warriors, administrators, and other nobles from other Marathi groups.
        Chhatrapati Shivaji Maharaj’s monarchy, referred to as the Maratha Kingdom, expanded into a large realm in the 18th century under the leadership of Peshwa Bajirao I. Marathas from the time of Shahu I recognised the Mughal emperor as their nominal suzerain, similar to other contemporary Indian entities, though in practice, Mughal politics were largely controlled by the Marathas between 1737 and 1803.

        After Aurangzeb's death in 1707, Chhatrapati Shivaji Maharaj's grandson Shahu under the leadership of Peshwa Bajirao revived Maratha power and confided a great deal of authority to the Bhat family, who became hereditary peshwas (prime ministers). 
        After he died in 1749, they became the effective rulers. The leading Maratha families – Scindia, Holkar, Bhonsle, and Gaekwad – extended their conquests in northern and central India and became more independent. The Marathas' rapid expansion was halted with the defeat of Panipat in 1761, at the hands of the Durrani Empire. However, the Marathas managed to retake most of their lost territories ten years later under the leadership of Peshwa Madhavrao I.[28] His death eventually marked the end of Peshwa’s effective authority over other chiefs in the empire. After he was defeated by the Holkar dynasty in 1802, the Peshwa Baji Rao II sought protection from the British East India Company, whose intervention destroyed the confederacy by 1818 after the Second and Third Anglo-Maratha Wars.

        The structure of the Maratha state was that of a confederacy of four rulers under the leadership of the Peshwa at Poona (now Pune) in western India. These were the Scindia, the Gaekwad based in Baroda, the Holkar based in Indore and the Bhonsle based in Nagpur. The stable borders of the confederacy after the Battle of Bhopal in 1737 extended from modern-day Maharashtra in the south to Gwalior in the north, to Orissa in the east or about a third ( 2.5 million km2) of the subcontinent.
        """
    )
]

print(raw_documents)


[Document(metadata={}, page_content='\n        The Maratha Empire, also referred to as the Maratha Confederacy, was an early modern polity in the Indian subcontinent. For most of its existence, it comprised the realms of the Peshwa and four major independent Maratha states under the nominal leadership of the former and nominal loyalty to the Chhatrapatis who were successors of Chhatrapati Shivaji Maharaj.\n\n        The Marathas were a Marathi-speaking warrior group from the western Deccan Plateau (present-day Maharashtra) that rose to prominence under leadership of Shivaji (17th century), who revolted against the Bijapur Sultanate and the Mughal Empire for establishing "Hindavi Swarajya" (lit.\u2009\'self-rule of Hindus\').\n\n        The religious attitude of Emperor Aurangzeb estranged non-Muslims, and the Maratha insurgency came at a great cost for his men and treasury. The Maratha government also included warriors, administrators, and other nobles from other Marathi groups.\n     

In [95]:
# Define chunking strategy
text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=24)
documents = text_splitter.split_documents(raw_documents[:3])
print(documents)

[Document(metadata={}, page_content='\n        The Maratha Empire, also referred to as the Maratha Confederacy, was an early modern polity in the Indian subcontinent. For most of its existence, it comprised the realms of the Peshwa and four major independent Maratha states under the nominal leadership of the former and nominal loyalty to the Chhatrapatis who were successors of Chhatrapati Shivaji Maharaj.\n\n        The Marathas were a Marathi-speaking warrior group from the western Deccan Plateau (present-day Maharashtra) that rose to prominence under leadership of Shivaji (17th century), who revolted against the Bijapur Sultanate and the Mughal Empire for establishing "Hindavi Swarajya" (lit.\u2009\'self-rule of Hindus\').\n\n        The religious attitude of Emperor Aurangzeb estranged non-Muslims, and the Maratha insurgency came at a great cost for his men and treasury. The Maratha government also included warriors, administrators, and other nobles from other Marathi groups.\n     

In [96]:
llm_transformer = LLMGraphTransformer(llm=chat)
graph_documents = llm_transformer.convert_to_graph_documents(documents)

In [97]:
# store to neo4j
res = kg.add_graph_documents(
    graph_documents,
    include_source=True,
    baseEntityLabel=True,
)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('apoc.create.addLabels' has been replaced by 'Cypher's dynamic labels; `SET n:$(labels)`.')} {position: line: 1, column: 257, offset: 256} for query: "MERGE (d:Document {id:$document.metadata.id}) SET d.text = $document.page_content SET d += $document.metadata WITH d UNWIND $data AS row MERGE (source:`__Entity__` {id: row.id}) SET source += row.properties MERGE (d)-[:MENTIONS]->(source) WITH source, row CALL apoc.create.addLabels( source, [row.type] ) YIELD node RETURN distinct 'done' AS result"
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future ve

In [ ]:
# Use below cyper command to delete the Graphs present in neo4j
# MATCH (n) DETACH DELETE n 


In [98]:
# Hybrid Retrieval for RAG
# create vector index
vector_index = Neo4jVector.from_existing_graph(
    OpenAIEmbeddings(),
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)

In [99]:
# Extract entities from text
class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )

In [100]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)

In [101]:
entity_chain = prompt | chat.with_structured_output(Entities)

In [102]:
# Test it out:
res = entity_chain.invoke(
    {"question": "After Aurangzeb's death in 1707, Chhatrapati Shivaji Maharaj's grandson Shahu under the leadership of Peshwa Bajirao revived Maratha power and confided a great deal of authority to the Bhat family, who became hereditary peshwas (prime ministers)."}
).names
print(res)

['Aurangzeb', 'Chhatrapati Shivaji Maharaj', 'Shahu', 'Peshwa Bajirao', 'Bhat family']


In [103]:
# Retriever
kg.query("CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")


[]

In [104]:
def generate_full_text_query(input: str) -> str:
    """
    Generate a full-text search query for a given input string.

    This function constructs a query string suitable for a full-text search.
    It processes the input string by splitting it into words and appending a
    similarity threshold (~2 changed characters) to each word, then combines
    them using the AND operator. Useful for mapping entities from user questions
    to database values, and allows for some misspelings.
    """
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()

In [105]:
# Fulltext index query
def structured_retriever(question: str) -> str:
    """
    Collects the neighborhood of entities mentioned
    in the question
    """
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        print(f" Getting Entity: {entity}")
        response = kg.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        # print(response)
        result += "\n".join([el["output"] for el in response])
    return result


In [106]:
print(structured_retriever("Who is Chhatrapati Shivaji Maharaj?"))

 Getting Entity: Chhatrapati Shivaji Maharaj


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }} {position: line: 3, column: 13, offset: 104} for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node,score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n              MATCH (node)<-[r:!MENTIONS]-(neighbor)\n              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output\n            }\n            RETURN output LIMIT 50\n            "


Chhatrapati Shivaji Maharaj - ALSO_KNOWN_AS -> Shivaji
Chhatrapati Shivaji Maharaj - LEADERSHIP -> Peshwa Bajirao I


In [107]:
# Final retrieval step
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [
        el.page_content for el in vector_index.similarity_search(question)
    ]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    print(f"\nFinal Data::: ==>{final_data}")
    return final_data

In [108]:
# Define the RAG chain
# Condense a chat history and follow-up question into a standalone question
_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

In [109]:
CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

In [110]:
def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

In [111]:
_search_query = RunnableBranch(
    # If input includes chat_history, we condense it with the follow-up question
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),  # Condense follow-up question and chat into a standalone_question
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | ChatOpenAI(temperature=0)
        | StrOutputParser(),
    ),
    # Else, we have no chat history, so just pass through the question
    RunnableLambda(lambda x: x["question"]),
)

In [112]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

In [113]:
chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | chat
    | StrOutputParser()
)

In [114]:
# TEST it all out!
res_simple = chain.invoke(
    {
        "question": "When did Aurangzeb die?",
    }
)

# print(f"\n Results === {res_simple}\n\n")

Search query: When did Aurangzeb die?
 Getting Entity: Aurangzeb


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }} {position: line: 3, column: 13, offset: 104} for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node,score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n              MATCH (node)<-[r:!MENTIONS]-(neighbor)\n              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output\n            }\n            RETURN output LIMIT 50\n            "
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.S


Final Data::: ==>Structured data:
Aurangzeb - ESTRANGED_NON_MUSLIMS -> Maratha Empire
Unstructured data:

text: 
        The Maratha Empire, also referred to as the Maratha Confederacy, was an early modern polity in the Indian subcontinent. For most of its existence, it comprised the realms of the Peshwa and four major independent Maratha states under the nominal leadership of the former and nominal loyalty to the Chhatrapatis who were successors of Chhatrapati Shivaji Maharaj.

        The Marathas were a Marathi-speaking warrior group from the western Deccan Plateau (present-day Maharashtra) that rose to prominence under leadership of Shivaji (17th century), who revolted against the Bijapur Sultanate and the Mughal Empire for establishing "Hindavi Swarajya" (lit. 'self-rule of Hindus').

        The religious attitude of Emperor Aurangzeb estranged non-Muslims, and the Maratha insurgency came at a great cost for his men and treasury. The Maratha government also included warriors, ad

In [115]:
res_hist = chain.invoke(
    {
        "question": "Who founded the Maratha Empire?",
        "chat_history": [
            ("Who founded the Maratha Empire?", "Chhatrapati Shivaji Maharaj founded the Maratha Empire.")
        ],
    }
)

print(f"\n === {res_hist}\n\n")


Search query: Who founded the Maratha Empire?
 Getting Entity: Maratha Empire


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }} {position: line: 3, column: 13, offset: 104} for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node,score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n              MATCH (node)<-[r:!MENTIONS]-(neighbor)\n              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output\n            }\n            RETURN output LIMIT 50\n            "
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.S


Final Data::: ==>Structured data:
Maratha Empire - ALSO_REFERRED_TO_AS -> Maratha Confederacy
Maratha Empire - COMPRISED_OF -> Peshwa
Maratha Empire - NOMINAL_LEADERSHIP -> Chhatrapatis
Maratha Empire - DEFEATED_BY -> Durrani Empire
Maratha Empire - RETREATED_UNDER_LEADERSHIP -> Peshwa
Maratha Empire - LEADERSHIP -> Peshwa Madhavrao I
Maratha Empire - PART_OF_CONFEDERACY -> Scindia
Maratha Empire - PART_OF_CONFEDERACY -> Holkar
Maratha Empire - PART_OF_CONFEDERACY -> Bhonsle
Maratha Empire - PART_OF_CONFEDERACY -> Gaekwad
Maratha Empire - EXTENDS_TO -> Maharashtra
Maratha Empire - EXTENDS_TO -> Gwalior
Maratha Empire - EXTENDS_TO -> Orissa
Maratha Empire - EXTENDED_TO -> Orissa
Maratha Empire - DEFEATED_AT_PANIPAT -> Durrani Empire
Maratha Empire - RETAKE_TERRITORIES -> Peshwa
Aurangzeb - ESTRANGED_NON_MUSLIMS -> Maratha Empire
Peshwa Bajirao I - EFFECTIVE_RULERS -> Maratha Empire
Scindia - EXTENDED_CONQUESTS -> Maratha Empire
Holkar - EXTENDED_CONQUESTS -> Maratha Empire
Bhonsle - EX